# Dokumentähnlichkeit mit Sentence Transformers

In diesem Notebook trainieren wir ein Sentence-Transformer-Modell, das semantisch ähnliche Fragen und Antworten in einem gemeinsamen Vektorraum abbildet. Als Trainingsdaten verwenden wir die deutsche Version des Natural-Questions-Datensatzes.

## Umgebung vorbereiten

Zunächst legen wir fest, welche GPU für das Training verwendet werden soll. Die Angabe `0` wählt die erste verfügbare CUDA-GPU aus.

In [27]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Daten vorbereiten

Der deutsche Natural-Questions-Datensatz liegt als komprimierte JSONL-Datei vor. Die Hilfsfunktion entfernt nicht benötigte Spalten, benennt Frage und Antwort einheitlich und reserviert 1.000 Beispiele für die Auswertung.

In [28]:
from datasets import load_dataset

def load_nq_german(data_file = "./data/ng_german.jsonl.gz"):
    # JSONL-Datei als Datensatz laden
    dataset = (
        load_dataset("json", data_files=data_file, split="train",num_proc=8)
        .remove_columns(["query", "answer"])
        .rename_column("question_de", "query")
        .rename_column("answer_de", "answer")
    )
    dataset_dict = dataset.train_test_split(test_size=1_000, seed=12)
    return dataset_dict


In [29]:
import logging
import random

import numpy
import torch
#from torch import mps  # noqa: F401
#torch.mps.device = mps
from datasets import Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [30]:
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)
random.seed(12)
torch.manual_seed(12)
numpy.random.seed(12)

## Modell konfigurieren

Wir verwenden ein mehrsprachiges Sentence-Transformer-Modell als Ausgangspunkt. Die Zufallswerte aus der vorherigen Zelle machen die Experimente besser reproduzierbar. Über die beiden Schalter lässt sich steuern, ob Rollen-Prompts verwendet und beim Pooling berücksichtigt werden.

In [31]:
# Diese Variablen können angepasst werden:
use_prompts = True
include_prompts_in_pooling = True

# 1. Zu trainierendes Modell und optionale Model-Card-Daten festlegen
base_model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

In [32]:
model = SentenceTransformer(
    base_model_name,
    #tokenizer_kwargs={"max_seq_length": 512},
    model_card_data=SentenceTransformerModelCardData(
        language="de",
        license="apache-2.0",
        model_name=f"{base_model_name} trained on german Natural Questions pairs",
    ),
).to(torch.bfloat16)

2026-09-09 15:23:43 - No device provided, using mps
2026-09-09 15:23:43 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-09 15:23:44 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-mpnet-base-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22f7640f94e81bb7f4f04daf1668850b38763a13d9%22 "HTTP/1.1 200 OK"
2026-09-09 15:23:44 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-09 15:23:44 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-09-09 15:23:45 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-09 15:23:45 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-09 15:23:45 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-09 15:23:46 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-09 15:23:46 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-09 15:23:46 - HTTP Request: HEAD https://hugging

In [33]:
model.set_pooling_include_prompt(include_prompts_in_pooling)

## Prompts definieren

Die Präfixe kennzeichnen, ob ein Text eine Suchanfrage oder ein Dokument ist. Dadurch kann das Modell beide Rollen beim Erzeugen der Embeddings unterscheiden.

In [34]:
# 2. Optional: Prompts definieren
if use_prompts:
    query_prompt = "query: "
    corpus_prompt = "document: "
    prompts = {
        "query": query_prompt,
        "answer": corpus_prompt,
    }

## Trainingsdaten erzeugen

Nun laden wir den Datensatz mit der zuvor definierten Hilfsfunktion und teilen ihn in Trainings- und Evaluierungsdaten auf.

In [35]:
# 3. Datensatz für das Training laden
dataset_dict = load_nq_german()
train_dataset: Dataset = dataset_dict["train"]
eval_dataset: Dataset = dataset_dict["test"]


2026-09-09 15:23:51 - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"


In [40]:
from collections import defaultdict
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator

# Retrieval-Daten für Top-k-Accuracy vorbereiten
eval_queries = {
    str(index): query
    for index, query in enumerate(eval_dataset["query"])
}
eval_corpus = {
    str(index): answer
    for index, answer in enumerate(eval_dataset["answer"])
}

answer_to_corpus_ids = defaultdict(set)
for corpus_id, answer in eval_corpus.items():
    answer_to_corpus_ids[answer].add(corpus_id)

eval_relevant_docs = {
    query_id: answer_to_corpus_ids[eval_corpus[query_id]]
    for query_id in eval_queries
}

retrieval_evaluator = InformationRetrievalEvaluator(
    queries=eval_queries,
    corpus=eval_corpus,
    relevant_docs=eval_relevant_docs,
    name="nq-german",
    accuracy_at_k=[1, 3, 5],
    mrr_at_k=[5],
    ndcg_at_k=[5],
    precision_recall_at_k=[5],
    map_at_k=[5],
    query_prompt=query_prompt if use_prompts else None,
    corpus_prompt=corpus_prompt if use_prompts else None,
    batch_size=64,
)

## Verlustfunktion auswählen

`CachedMultipleNegativesRankingLoss` vergleicht passende Frage-Antwort-Paare mit den übrigen Beispielen eines Batches. Das Caching erlaubt dabei eine größere effektive Batch-Größe bei begrenztem GPU-Speicher.

In [37]:
# 4. Verlustfunktion definieren
#loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16) 
loss = MultipleNegativesRankingLoss(model) # <- funktioniert auch mit MPS (Apple Silicon)


## Training konfigurieren

Für das Training legen wir unter anderem die Anzahl der Epochen, Batch-Größen, Lernrate und Auswertungsintervalle fest. Eine Viertel-Epoche verkürzt hier die Laufzeit während der Entwicklung.

In [ ]:
# 5. Optionale Trainingsparameter festlegen

# TODO: Größe des Trainingsdatensatzes für Tests begrenzen

run_name = "nq-german-" + base_model_name.split("/")[-1]
if use_prompts:
    run_name += "-prompts"
if not include_prompts_in_pooling:
    run_name += "-exclude-pooling-prompts"
args = SentenceTransformerTrainingArguments(
    # Erforderlicher Parameter:
    output_dir=f"models/{run_name}",
    # Optionale Trainingsparameter:
    num_train_epochs=0.25, # Training während der Entwicklung auf eine Viertel-Epoche begrenzen
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=4e-5,
    warmup_ratio=0.1,   
    fp16=False,  # Auf False setzen, falls die GPU kein FP16 unterstützt
    bf16=True,  # Auf True setzen, falls die GPU BF16 unterstützt
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # Die Verlustfunktion profitiert von Batches ohne Duplikate
    # Optionale Parameter für Protokollierung und Debugging:
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy="steps",
    save_steps=0.5,
    save_total_limit=2,
    logging_steps=5,
    logging_first_step=True,
    run_name=run_name,  # Wird von W&B verwendet, falls `wandb` installiert ist
    seed=12,
    prompts=prompts if use_prompts else None,
)

2026-09-09 15:23:51 - The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


## Das Training \o/

Jetzt verbinden wir Modell, Trainingsparameter, Datensätze, Verlustfunktion und Retrieval-Evaluator in einem Trainer. Während des Trainings werden neben dem Eval-Loss auch Top-1-, Top-3- und Top-5-Accuracy berechnet.

In [39]:
# 6. Trainer erstellen und Training starten
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=retrieval_evaluator,
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/Users/ogu/source/workshops/workshop-ki-deepdive/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


2026-09-09 15:28:03 - Information Retrieval Evaluation of the model on the nq-german dataset in epoch 0.12516129032258064 after 97 steps:


ValueError: max() iterable argument is empty

In [ ]:
# 7. Trainiertes Modell speichern
model.save_pretrained(f"models/{run_name}/final")

2026-09-09 14:43:03 - Saving model to models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Modell testen

Wir laden das gespeicherte Modell, erzeugen Embeddings für einige Beispielsätze und berechnen deren paarweise Ähnlichkeit. Hohe Werte zeigen an, dass zwei Texte im Vektorraum nahe beieinanderliegen.

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Vortrainiertes Sentence-Transformer-Modell laden
model = SentenceTransformer(f"./models/{run_name}/final")

# Zu kodierende Sätze
sentences = [
    "query: Das ist eine Frage über Kekse.",
    "answer: Das hier ist ein Keksrezept",
    "query: Ich mag Möven, oder?",
    "answer: Ich bin Paul die Möve",
]

# 2. Embeddings mit model.encode() berechnen
embeddings = model.encode(sentences)


similarities = model.similarity(embeddings, embeddings)
print(similarities)

2026-09-09 14:43:07 - No device provided, using mps
2026-09-09 14:43:07 - Loading SentenceTransformer model from ./models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/final.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[1.0000, 0.8437, 0.2214, 0.0844],
        [0.8437, 1.0000, 0.2122, 0.2014],
        [0.2214, 0.2122, 1.0000, 0.3226],
        [0.0844, 0.2014, 0.3226, 1.0000]])


In [ ]:
import csv
import numpy as np

sample_size = 5_000
max_text_length = 500

short_pairs = train_dataset.filter(
    lambda example: (
        0 < len(example["query"].strip()) <= max_text_length
        and 0 < len(example["answer"].strip()) <= max_text_length
    )
)



if len(short_pairs) < sample_size:
    raise ValueError(f"Nur {len(short_pairs)} kurze Frage-Antwort-Paare gefunden.")


sampled_pairs = short_pairs.shuffle(seed=12).select(range(sample_size))

queries = [" ".join(query.split()) for query in sampled_pairs["query"]]
answers = [" ".join(answer.split()) for answer in sampled_pairs["answer"]]

query_prompt_name = "query" if "query" in model.prompts else None
answer_prompt_name = "answer" if "answer" in model.prompts else None



query_embeddings = model.encode(queries, prompt_name=query_prompt_name, batch_size=64, show_progress_bar=True)
answer_embeddings = model.encode(answers, prompt_name=answer_prompt_name, batch_size=64, show_progress_bar=True)




# Frage und zugehörige Antwort im Projector direkt nebeneinander anordnen.
projector_embeddings = np.stack((query_embeddings, answer_embeddings), axis=1).reshape(2 * sample_size, -1)

np.savetxt("projector_embeddings_1000_pairs.tsv", projector_embeddings, delimiter="\t", fmt="%.8g")



with open("projector_metadata_1000_pairs.tsv", "w", encoding="utf-8", newline="") as metadata_file:
    writer = csv.writer(metadata_file, delimiter="\t", lineterminator="\n")
    writer.writerow(["pair_id", "text"])
    for pair_id, (query, answer) in enumerate(zip(queries, answers), start=1):
        writer.writerow([pair_id, query])
        writer.writerow([pair_id, answer])

print(f"{sample_size} Paare / {len(projector_embeddings)} Embeddings exportiert.")

Filter:   0%|          | 0/99188 [00:00<?, ? examples/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

5000 Paare / 10000 Embeddings exportiert.


## Embeddings visualisieren

Für den [TensorFlow Embedding Projector](https://projector.tensorflow.org/) speichern wir die Embeddings und ihre Beschriftungen in zwei tabulatorgetrennten Dateien. Dort lassen sich die Vektoren interaktiv in zwei oder drei Dimensionen untersuchen.

In [ ]:
import numpy as np
# Dateien für den TensorFlow Embedding Projector konvertieren

# Als TSV speichern
np.savetxt('output.tsv', embeddings, delimiter='\t', fmt='%g')

with open("description.tsv","w") as f:
    f.writelines(sentence + "\n" for sentence in sentences)

# Tutorial:

Das Modell kann nun semantisch ähnliche Fragen und Antworten erkennen. Machen Sie sich mit dem Notebook vertraut und untersuchen Sie, wie sich verschiedene Einstellungen auf die Ähnlichkeitswerte auswirken.

**Ihre Aufgabe ist es, die Qualität der Dokumentähnlichkeit zu verbessern.**

Hier sind einige Ideen:

* Vergleichen Sie das Training mit und ohne Rollen-Prompts. Prüfen Sie auch, welchen Einfluss `include_prompts_in_pooling` hat.

* Testen Sie andere mehrsprachige oder deutschsprachige Modelle aus dem [Model Hub](https://huggingface.co/models?library=sentence-transformers).

* Variieren Sie Lernrate, Batch-Größe und Anzahl der Epochen. Bewerten Sie Änderungen nicht nur mit einzelnen Beispielen, sondern mit einem geeigneten Evaluierungsdatensatz.

* Ergänzen Sie eigene Frage-Antwort-Paare und betrachten Sie deren Position im TensorFlow Embedding Projector.